# Modul 14 · Projektwerkstatt zu zweit
**Data Science 1** · linkit courses

**Leitfrage:** Wie wird aus zwei Analysen eine Entscheidungsvorlage, die die Abnahme besteht?

Diese Vorlage ist euer Startpunkt für das Abschlussprojekt. Sie führt euch in fünf Schritten vom Datenpaket zur Entscheidungsvorlage: Daten prüfen, Analyseplan festhalten, Analyse 1 (Vorhersage), Analyse 2 (Wirkung), Entscheidung. Sie rechnet beide Tracks: Track A mit der Datenfabrik Karlsruhe e.V. (Erinnerungen und Anmeldegrenzen) und Track B mit der Nordwerk GmbH (Rollout des Warnsystems). Jede Funktion ist ein Werkzeug, jede Vorgabe ein Platzhalter. Eine Musterlösung gibt es nicht.

**So arbeitet ihr**
- **Schritt 0:** Stellt in der Einstellungs-Zelle euren Track ein und lasst die Vorlage einmal mit `PAKET = 0` von oben nach unten laufen (Übungspaket 00). Das Projekthandbuch und die Lerneinheiten rechnen mit demselben Paket, ihr könnt jede Zahl dort nachvollziehen.
- Danach stellt ihr `PAKET` auf eure Nummer und arbeitet nur noch mit eurem Paket. Jedes Paket hat eigene Datenfehler. Die Bereinigung der Vorlage kennt nur die Fehler des Übungspakets.
- Alles mit „▶“ ist ein Platzhalter, den ihr begründet ersetzt: Prüfregeln, Bereinigung, Analyseplan, Merkmale, Annahmen der Entscheidung. Haltet jede Entscheidung mit Grund in `ENTSCHEIDUNGEN.md` fest.
- Im Browser speichert die Kursseite eure Änderungen nur in diesem Browser. Für euer Repository ladet ihr das Notebook herunter oder arbeitet in Google Colab mit einer eigenen Kopie (*Datei → Kopie in Drive speichern*).
- Der erste Import lädt im Browser SciPy, statsmodels und scikit-learn, das dauert einmalig etwa 15 Sekunden. Jede weitere Zelle braucht wenige Sekunden.
- Aller Zufall hängt an `SEED` in der Einstellungs-Zelle: Bootstrap und Simulationen legen ihren Generator mit `np.random.default_rng(seed)` an, die logistische Regression bekommt `random_state=SEED`. Im Abnahmegespräch kann Erik Schulze den Seed oder eine Annahme ändern und eine Zelle neu laufen lassen.
- Alle Personen, Maschinen und Daten in den Paketen sind erfunden. Gebt Ergebnisse nur für Gruppen an, nie für einzelne Personen.

In [ ]:
# Einstellungen: Track, Paketnummer und Seed. Schritt 0: einmal mit PAKET = 0 von oben nach unten laufen lassen.
TRACK = "B"   # ▶ "A" für die Datenfabrik Karlsruhe e.V., "B" für die Nordwerk GmbH
PAKET = 0     # ▶ eure Paketnummer von 1 bis 20; 0 ist das Übungspaket
SEED = 14     # aller Zufall der Vorlage hängt an diesem Seed

In [ ]:
# Daten laden: Auf der Kursseite liegen die Übungspakete schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["projekt/A-00.zip", "projekt/B-00.zip"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten/projekt", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten/projekt"))))

## Paket laden

`hole_paket(track, paket)` liest das Paket direkt aus dem Zip: jede CSV-Datei als Tabelle, ohne Spalten umzuwandeln. Das Ergebnis `roh` ist ein Wörterbuch, etwa `roh["maschinentage"]` oder `roh["anmeldungen"]`. Fehlt das Paket, holt die Funktion es selbst, im Browser von der Kursseite und in Colab aus dem Repository des Kurses. Lest die ganze `LIESMICH.md` eures Pakets: Sie enthält den Auftrag, die Spalten und die Annahmen der Auftraggeber. Aus ihr leitet ihr in Schritt 1 die Prüfregeln ab.

In [ ]:
import io, sys, json, zipfile, hashlib, datetime, urllib.parse, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Im Browser kann threadpoolctl beim Start von scikit-learn eine Meldung der Plattform zeigen („JsProxy.as_object_map() is deprecated“).
# Sie betrifft eure Rechnung nicht. Dieser Filter blendet genau diese eine Meldung aus.
warnings.filterwarnings("ignore", message=".*as_object_map.*")
print("Bereit.")

In [ ]:
def hole_paket(track, paket):
    """Liest das Paket track-NN als Wörterbuch {Tabellenname: DataFrame}; fehlt die Datei, holt die Funktion sie selbst."""
    if track not in ("A", "B") or not 0 <= int(paket) <= 20:
        raise ValueError(f"TRACK muss \"A\" oder \"B\" sein und PAKET eine Zahl von 0 bis 20 (eingestellt: {track!r}, {paket!r}).")
    name = f"{track}-{int(paket):02d}"
    pfad = f"daten/projekt/{name}.zip"
    if not os.path.exists(pfad):
        if sys.platform == "emscripten":                 # im Browser: von der Kursseite
            import js, requests
            antwort = requests.get(urllib.parse.urljoin(str(js.location.href), f"daten/ds1/projekt/{name}.zip"))
            antwort.raise_for_status()
            with open(pfad, "wb") as f:
                f.write(antwort.content)
        else:                                            # in Colab oder lokal: aus dem Repository
            urllib.request.urlretrieve(QUELLE + f"projekt/{name}.zip", pfad)
    tabellen = {}
    with zipfile.ZipFile(pfad) as z:
        liesmich = z.read(f"{name}/LIESMICH.md").decode("utf-8")
        for datei in sorted(z.namelist()):
            if datei.endswith(".csv"):
                tabellen[datei.split("/")[-1][:-4]] = pd.read_csv(z.open(datei))
    print("\n".join(liesmich.strip().split("\n")[:5]))
    print("…\n")
    for tabelle, df in tabellen.items():
        print(f"{tabelle}: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
    return tabellen


roh = hole_paket(TRACK, PAKET)

## Schritt 1 · Euer Paket laden und prüfen

*Ergebnis:* Eure Prüfregeln aus der `LIESMICH.md` stehen in der Vorlage, der Datenqualitätsbericht nennt jedes gefundene Problem mit Umfang und Umgang.

Zuerst ein **Steckbrief** je Tabelle: Typ, fehlende Werte, Zahl verschiedener Werte, Minimum, Maximum und Beispielwerte je Spalte. Achtet auf Zahlenspalten mit dem Typ `object` (dort steht irgendwo Text), auf Minima und Maxima außerhalb des Datenblatts und auf Kennungen, die anders aussehen als beschrieben.

In [ ]:
def steckbrief(df):
    """Je Spalte: Typ, fehlend, eindeutig, Minimum, Maximum und drei Beispielwerte."""
    zeilen = []
    for spalte in df.columns:
        x = df[spalte]
        werte = x.dropna()
        try:
            kleinster, groesster = werte.min(), werte.max()
        except TypeError:                                # Zahlen und Text gemischt
            kleinster, groesster = "gemischt", "gemischt"
        zeilen.append({"Spalte": spalte, "Typ": str(x.dtype), "fehlend": int(x.isna().sum()), "eindeutig": int(x.nunique()),
                       "Minimum": kleinster, "Maximum": groesster, "Beispiele": ", ".join(str(w) for w in werte.unique()[:3])})
    return pd.DataFrame(zeilen).set_index("Spalte")


for tabelle, df in roh.items():
    print(f"\n{tabelle} ({len(df)} Zeilen)")
    print(steckbrief(df).to_string())

**Prüfregeln** machen Zusagen des Datenblatts prüfbar. Jede Regel bekommt einen Namen und eine boolesche Spalte, die für jede Zeile sagt, ob sie die Regel **verletzt**. `regel` zeigt ✅ oder ❌ mit der Zahl der verletzenden Zeilen und sammelt alles in `REGELN`. Die Vorlage bringt je Track drei Beispiele mit. Ein ❌ bricht nichts ab, es zeigt euch, wo ihr hinschauen müsst.

In [ ]:
REGELN = []


def regel(name, verletzt):
    """Prüfregel: verletzt ist eine boolesche Spalte (True = Zeile verletzt die Regel). Zeigt ✅ oder ❌ mit Anzahl."""
    anzahl = int(np.asarray(verletzt, dtype=bool).sum())
    REGELN.append({"Regel": name, "verletzt": anzahl})
    print(f"✅ {name}" if anzahl == 0 else f"❌ {name}: {anzahl} Zeilen verletzen die Regel")
    return anzahl


def als_zahl(x):
    """Spalte als Zahl; Text, der keine Zahl ist, wird fehlend (nur für Prüfregeln, die Bereinigung meldet jede Umwandlung)."""
    return pd.to_numeric(x, errors="coerce")


def regeln_A(t):
    """Drei Beispielregeln für Track A, abgeleitet aus der LIESMICH.md."""
    a = t["anmeldungen"]
    regel("anmeldungen: Kennung anmeldung ist eindeutig", a["anmeldung"].duplicated())
    regel("anmeldungen: person und veranstaltung stehen in personen.csv und veranstaltungen.csv",
          ~a["person"].isin(t["personen"]["person"]) | ~a["veranstaltung"].isin(t["veranstaltungen"]["veranstaltung"]))
    regel("anmeldungen: erschienen ist 0 oder 1", ~als_zahl(a["erschienen"]).isin([0, 1]))


def regeln_B(t):
    """Drei Beispielregeln für Track B, abgeleitet aus der LIESMICH.md."""
    d = t["maschinentage"]
    regel("maschinentage: je Maschine und Tag genau eine Zeile", d.duplicated(["datum", "maschine"]))
    regel("maschinentage: Kennung der Maschine steht in maschinen.csv (Form M-NN)", ~d["maschine"].isin(t["maschinen"]["maschine"]))
    zahl = als_zahl(d["stillstand_min"])
    regel("maschinentage: stillstand_min ist eine Zahl und ein Vielfaches von 5", zahl.isna() | (zahl % 5 != 0))

In [ ]:
def eure_regeln(t):
    """▶ Eure Prüfregeln. Geht die LIESMICH.md Satz für Satz durch: Jede Zusage über Kennungen, Wertebereiche,
    Einheiten, Zeiträume oder Zusammenhänge zwischen Tabellen wird eine Regel."""
    if TRACK == "A":
        # Beispiel: Die LIESMICH.md sagt „catering: 0 oder 1, 1, wenn es Essen gab“. Aus jeder solchen Zusage wird eine Regel,
        # die für jede Zeile True liefert, wenn sie der Zusage widerspricht:
        # regel("veranstaltungen: catering ist 0 oder 1", ~als_zahl(t["veranstaltungen"]["catering"]).isin([0, 1]))
        pass
    else:
        # Beispiel: Die LIESMICH.md sagt „laufzeit_h: 24 an Werktagen, 8 an Samstagen mit Frühschicht“. Aus jeder solchen Zusage
        # wird eine Regel, die für jede Zeile True liefert, wenn sie der Zusage widerspricht:
        # regel("maschinentage: laufzeit_h ist 24 oder 8", ~als_zahl(t["maschinentage"]["laufzeit_h"]).isin([24, 8]))
        pass

In [ ]:
# Prüfregeln auf den Rohdaten
REGELN.clear()
regeln_A(roh) if TRACK == "A" else regeln_B(roh)
eure_regeln(roh)

Jetzt die **Bereinigung**. Jede Korrektur meldet mit `melde(problem, umfang, umgang)`, was sie gefunden und wie sie es behandelt hat, auch wenn sie nichts ändern musste. Die Meldungen sammelt `BERICHT`, und `bericht()` zeigt sie als **Datenqualitätsbericht**. Er gehört in euer Repository und in die Abnahme.

`bereinige_A` und `bereinige_B` behandeln genau die Fehlerarten, die eine sorgfältige Prüfung im Übungspaket ihres Tracks findet. **Diese Bereinigung kennt nur das Übungspaket. Euer Paket hat andere Fehler.** Mit eurem Paket meldet sie vermutlich oft „0 Zeilen“, und eure Prüfregeln zeigen euch, was noch fehlt. Ergänzt die Bereinigung, statt sie blind zu übernehmen, und meldet jede eigene Korrektur ebenfalls mit `melde`.

In [ ]:
BERICHT = []


def melde(problem, umfang, umgang):
    """Ein Eintrag im Datenqualitätsbericht: was, wie viel, wie behandelt."""
    BERICHT.append({"Problem": problem, "Umfang": umfang, "Umgang": umgang})
    print(f"• {problem}: {umfang} → {umgang}")


def bericht():
    """Zeigt den Datenqualitätsbericht als Tabelle."""
    print(pd.DataFrame(BERICHT, columns=["Problem", "Umfang", "Umgang"]).to_string(index=False))


def als_datum(x, was):
    """Text JJJJ-MM-TT als Datum; was sich nicht lesen lässt, wird fehlend und gemeldet."""
    datum = pd.to_datetime(x, format="%Y-%m-%d", errors="coerce")
    melde(f"{was} nicht als Datum JJJJ-MM-TT lesbar", f"{int((datum.isna() & x.notna()).sum())} Zeilen", "als fehlend gesetzt")
    return datum

In [ ]:
# ▶ Diese Bereinigung kennt nur das Übungspaket. Euer Paket hat andere Fehler.
def bereinige_A(roh):
    """Bereinigung für Track A, so wie sie das Übungspaket A-00 braucht. Jede Korrektur meldet ihren Umfang."""
    p, v, a = roh["personen"].copy(), roh["veranstaltungen"].copy(), roh["anmeldungen"].copy()
    # erschienen als Text „ja“ und „nein“
    text = a["erschienen"].astype(str).str.strip().str.lower()
    ja_nein = text.isin(["ja", "nein"])
    a["erschienen"] = pd.to_numeric(a["erschienen"].where(~ja_nein, text.map({"ja": 1, "nein": 0})), errors="coerce")
    melde("erschienen als Text „ja“ oder „nein“", f"{int(ja_nein.sum())} Zeilen", "in 1 und 0 umgesetzt")
    melde("erschienen weder 0 noch 1", f"{int((~a['erschienen'].isin([0, 1])).sum())} Zeilen", "als fehlend gesetzt")
    a["erschienen"] = a["erschienen"].where(a["erschienen"].isin([0, 1]))
    a["erinnerung"] = pd.to_numeric(a["erinnerung"], errors="coerce")
    # Datumsangaben
    v["datum"] = als_datum(v["datum"], "veranstaltungen.datum")
    a["angemeldet_am"] = als_datum(a["angemeldet_am"], "anmeldungen.angemeldet_am")
    # doppelte Anmeldungen: gleiche Person und Veranstaltung, die erste bleibt
    a = a.sort_values(["angemeldet_am", "anmeldung"])
    doppelt = a.duplicated(["person", "veranstaltung"])
    melde("doppelte Anmeldungen (gleiche Person und Veranstaltung)", f"{int(doppelt.sum())} Zeilen", "die erste behalten, die übrigen entfernt")
    a = a[~doppelt]
    # abgesagte Veranstaltungen: mindestens 8 Anmeldungen und niemand erschienen
    je_v = a.groupby("veranstaltung")["erschienen"].agg(["size", "sum"])
    abgesagt = je_v.index[(je_v["size"] >= 8) & (je_v["sum"] == 0)]
    melde("Veranstaltung mit mindestens 8 Anmeldungen und niemandem erschienen", f"{len(abgesagt)} ({', '.join(abgesagt) or 'keine'})",
          "als abgesagt ausgeschlossen")
    v, a = v[~v["veranstaltung"].isin(abgesagt)], a[~a["veranstaltung"].isin(abgesagt)]
    return {"personen": p.reset_index(drop=True), "veranstaltungen": v.reset_index(drop=True),
            "anmeldungen": a.sort_values("anmeldung").reset_index(drop=True)}

In [ ]:
# ▶ Diese Bereinigung kennt nur das Übungspaket. Euer Paket hat andere Fehler.
def bereinige_B(roh):
    """Bereinigung für Track B, so wie sie das Übungspaket B-00 braucht. Jede Korrektur meldet ihren Umfang."""
    m, d = roh["maschinen"].copy(), roh["maschinentage"].copy()
    # exakt doppelte Zeilen
    doppelt = d.duplicated()
    melde("exakt doppelte Zeilen", f"{int(doppelt.sum())} Zeilen", "entfernt")
    d = d[~doppelt].copy()
    # Kennungen auf die Form M-NN bringen
    alt = d["maschine"].astype(str)
    nummer = alt.str.extract(r"^\s*[Mm]\D*(\d+)\s*$")[0]
    neu = ("M-" + nummer.str.zfill(2)).where(nummer.notna(), alt)
    geaendert = neu != alt
    melde("Kennung nicht in der Form M-NN", f"{int(geaendert.sum())} Zeilen ({', '.join(sorted(alt[geaendert].unique())) or 'keine'})", "auf M-NN gebracht")
    d["maschine"] = neu
    d["datum"] = als_datum(d["datum"], "maschinentage.datum")
    m["pilot_start"] = pd.to_datetime(m["pilot_start"], format="%Y-%m-%d", errors="coerce")
    for spalte in ["vibration_mm_s", "lagertemp_c", "strom_pct", "stillstand_min"]:
        zahl = pd.to_numeric(d[spalte], errors="coerce")
        melde(f"{spalte} keine Zahl", f"{int((zahl.isna() & d[spalte].notna()).sum())} Zeilen", "als fehlend gesetzt")
        d[spalte] = zahl
    d = d.sort_values(["maschine", "datum"]).reset_index(drop=True)
    # stillstand_min kein Vielfaches von 5: Stunden statt Minuten im betroffenen Zeitraum der Maschine
    krumm = d["stillstand_min"].notna() & (d["stillstand_min"] % 5 != 0)
    for maschine, tage in d[krumm].groupby("maschine")["datum"]:
        zeitraum = (d["maschine"] == maschine) & d["datum"].between(tage.min(), tage.max())
        melde(f"stillstand_min von {maschine} kein Vielfaches von 5 ({tage.min():%d.%m.%Y} bis {tage.max():%d.%m.%Y})",
              f"{int(zeitraum.sum())} Tage im Zeitraum, {len(tage)} davon krumm", "als Stunden gelesen: mal 60, auf 5 gerundet")
        d.loc[zeitraum, "stillstand_min"] = (d.loc[zeitraum, "stillstand_min"] * 60 / 5).round() * 5
    if not krumm.any():
        melde("stillstand_min kein Vielfaches von 5", "0 Zeilen", "nichts zu tun")
    # Lücken: Produktionstage je Maschine
    tage = d.groupby("maschine")["datum"].nunique()
    alle = d["datum"].nunique()
    for maschine, n in tage[tage < alle].items():
        melde(f"Lücke: {maschine} mit {n} statt {alle} Produktionstagen", f"{alle - n} Tage", "nur Mittel je Maschinentag, nie Summen")
    if (tage == alle).all():
        melde("Lücken in den Produktionstagen", "0 Maschinen", "nichts zu tun")
    return {"maschinen": m, "maschinentage": d}

In [ ]:
# Bereinigen und den Bericht neu sammeln (bei erneutem Ausführen nichts doppelt)
BERICHT.clear()
daten = bereinige_A(roh) if TRACK == "A" else bereinige_B(roh)
# ▶ eure eigenen Korrekturen, jede mit melde(problem, umfang, umgang)

In [ ]:
# Dieselben Prüfregeln auf den bereinigten Daten, dann der Datenqualitätsbericht
REGELN.clear()
print("Prüfregeln nach der Bereinigung")
regeln_A(daten) if TRACK == "A" else regeln_B(daten)
eure_regeln(daten)
print()
bericht()

## Schritt 2 · Analyseplan festhalten

*Ergebnis:* Der Analyseplan steht im Notebook und als Commit im Repository, bevor ihr Ergebnisse der beiden Analysen anschaut.

Ein Analyseplan legt fest, was ihr rechnet, **bevor** ihr die Ergebnisse kennt. Dann kann niemand, auch ihr selbst nicht, die Methode passend zum Wunschergebnis wählen. Spätere Änderungen sind erlaubt, aber nur als gekennzeichneter Nachtrag mit Grund. Kopiert dieses Gerüst nach `ANALYSEPLAN.md`, füllt es aus und committet es zusammen mit dem Notebook.

```markdown
# Analyseplan · Track ▶ · Paket ▶ · ▶ Namen

Festgehalten am ▶ Datum, Prüfsumme des PLAN (SHA-256): ▶

## Frage und Entscheidung
▶ Welche Entscheidung trifft die Werkleitung oder der Vorstand mit eurer Vorlage? Welche Regel gilt (Schätzung oder Intervallgrenze)?

## Daten
▶ Paket, Zeitraum, Bereinigung (Verweis auf den Datenqualitätsbericht), ausgeschlossene Zeilen mit Grund

## Analyse 1 · Vorhersage (▶ Name)
▶ Zielgröße, Merkmale, zeitliche Grenze für den Test, Basis zum Vergleich, Gütemaß mit Intervall

## Analyse 2 · Wirkung (▶ Name)
▶ Design, Zeilen oder Zeiträume, Schätzer, Intervall, welche Annahme ihr wie prüft

## Entscheidungsrechnung
▶ Annahmen der Auftraggeber, wie die beiden Analysen eingehen, wie ihr die Unsicherheit weitergebt

## Nachträge
▶ Datum, Änderung, Grund
```

`PLAN` enthält die Vorgaben, die in Schritt 3 bis 5 die Rechnung steuern. **Jeder Eintrag mit „▶“ ist ein Platzhalter.** Prüft ihn an der `LIESMICH.md` und an den Daten, bevor ihr ihn übernehmt. `plan_festhalten(PLAN)` zeigt einen Zeitstempel und eine Prüfsumme (SHA-256) des Plans: Schreibt beide in `ANALYSEPLAN.md`. Ändert sich ein Wert im `PLAN`, ändert sich die Prüfsumme.

In [ ]:
def plan_festhalten(plan):
    """Zeigt den Plan mit Zeitstempel und SHA-256-Prüfsumme zum Committen."""
    text = json.dumps(plan, ensure_ascii=False, sort_keys=True, indent=1)
    pruefsumme = hashlib.sha256(text.encode("utf-8")).hexdigest()
    print("Analyseplan festgehalten am", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))
    print("Prüfsumme (SHA-256):", pruefsumme)
    print(text)
    return pruefsumme

In [ ]:
# Analyseplan: Jeder Eintrag mit ▶ ist ein Platzhalter. Prüft und ersetzt ihn begründet, bevor ihr Ergebnisse anschaut.
if TRACK == "A":
    PLAN = {
        "frage": "▶ Soll jede angemeldete Person eine Erinnerung bekommen, und wie viele Anmeldungen je Raumgröße?",
        "split": "2025-10-01",               # ▶ Grenze Training/Test: Bildet der Testzeitraum die Lage ab, für die ihr entscheidet?
        "zeilen": "datum >= '2024-10-01'",   # ▶ Anmeldungen für Analyse 2: Prüft in LIESMICH.md und Daten, wann und wie die Erinnerung zugeteilt wurde.
        "bloecke": False,                    # ▶ Prüft, auf welcher Ebene zugeteilt wurde und was das für die Standardfehler heißt.
        "kovariate": None,                   # ▶ eine Größe, die vor der Veranstaltung feststand und das Erscheinen vorhersagt, oder None
        "aufschlag": "Schätzung",            # ▶ welche Wirkung der Erinnerung in die Anmeldegrenze eingeht: "Schätzung" oder "obere Grenze"
        "regel": "▶ Erinnerung einführen, wenn die Wirkung mindestens 3 Prozentpunkte beträgt (Schätzung oder untere Intervallgrenze?)",
    }
else:
    PLAN = {
        "frage": "▶ Soll das Warnsystem auf weitere Maschinen ausgerollt werden, und auf welche?",
        "split": "2025-10-01",                          # ▶ Grenze Training/Test: Bildet der Testzeitraum die Lage ab, für die ihr entscheidet?
        "vorperiode": ("2025-10-01", "2025-12-31"),     # ▶ Vorperiode der Differenz-in-Differenzen: Prüft, wie die Pilotmaschinen ausgewählt wurden.
        "nach_ab": "2026-01-12",                        # Pilotstart laut maschinen.csv
        "regel": "▶ Rollout auf eine Maschine, wenn das Netto je Jahr positiv ist (Mittel oder untere Intervallgrenze?)",
    }
PRUEFSUMME = plan_festhalten(PLAN)

## Schritt 3 · Analyse 1: Vorhersage

*Ergebnis:* Modell mit zeitlichem Test, Vergleich mit einer einfachen Basis, AUC mit Intervall und eine Größe für die Entscheidung.

Die Werkzeuge dieser Analyse:
- `zeitlich_teilen(df, spalte, grenze)` teilt eine Tabelle an einem Datum in Training (davor) und Test (ab dem Datum). Ein zufälliger Split würde Wissen aus der Zukunft ins Training tragen.
- `auc_mit_intervall(y, p, seed)` gibt die AUC mit einem 95-%-Intervall aus 1.000 Bootstrap-Ziehungen des Testteils.
- `kalibrierung(y, p)` zeichnet vorhergesagte gegen beobachtete Häufigkeiten in zehn gleich großen Gruppen. Für Entscheidungen zählt, ob eine vorhergesagte Wahrscheinlichkeit von 30 % auch zu etwa 30 % passt.
- `merkmale_A(daten)` und `merkmale_B(daten)` bauen die Merkmale des Startmodells. Jedes Merkmal darf nur Wissen enthalten, das am Tag der Vorhersage schon vorlag.

In [ ]:
def zeitlich_teilen(df, spalte, grenze):
    """Training: Zeilen vor dem Datum grenze, Test: Zeilen ab grenze."""
    grenze = pd.Timestamp(grenze)
    return df[df[spalte] < grenze].copy(), df[df[spalte] >= grenze].copy()


def auc_mit_intervall(y, p, seed, ziehungen=1000):
    """AUC mit 95-%-Intervall aus einem Bootstrap über Zeilen (vektorisiert, Bindungen zählen halb)."""
    y, p = np.asarray(y, dtype=int), np.asarray(p, dtype=float)
    rng = np.random.default_rng(seed)
    n = len(y)
    reihenfolge = np.argsort(p, kind="stable")
    y, p = y[reihenfolge], p[reihenfolge]
    gruppen = np.flatnonzero(np.r_[True, np.diff(p) != 0])          # Beginn jeder Gruppe gleicher Werte
    werte = []
    for _ in range(ziehungen // 100):                                # je 100 Ziehungen auf einmal
        index = rng.integers(0, n, size=(100, n))
        anzahl = np.bincount((np.arange(100)[:, None] * n + index).ravel(), minlength=100 * n).reshape(100, n)
        pos = np.add.reduceat(anzahl * y, gruppen, axis=1)
        neg = np.add.reduceat(anzahl * (1 - y), gruppen, axis=1)
        unter = np.cumsum(neg, axis=1) - neg                         # Negative mit kleinerem Wert
        with np.errstate(invalid="ignore", divide="ignore"):
            werte.append(((pos * (unter + neg / 2)).sum(axis=1)) / (pos.sum(axis=1) * neg.sum(axis=1)))
    werte = np.concatenate(werte)
    unten, oben = np.nanquantile(werte, [0.025, 0.975])
    return float(roc_auc_score(y, p)), float(unten), float(oben)


def kalibrierung(y, p, gruppen=10):
    """Vorhergesagte gegen beobachtete Häufigkeit in gleich großen Gruppen; zeichnet die Abbildung und gibt die Tabelle zurück."""
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    grenzen = np.quantile(p, np.linspace(0, 1, gruppen + 1)[1:-1])
    gruppe = np.digitize(p, grenzen)
    tabelle = pd.DataFrame({"gruppe": gruppe, "vorhergesagt": p, "beobachtet": y}).groupby("gruppe").agg(
        vorhergesagt=("vorhergesagt", "mean"), beobachtet=("beobachtet", "mean"), anzahl=("beobachtet", "size"))
    fig, ax = plt.subplots(figsize=(5, 4.5), layout="constrained")
    oben = max(tabelle[["vorhergesagt", "beobachtet"]].max()) * 1.1
    ax.plot([0, oben], [0, oben], color="grey", linestyle="--", label="perfekt kalibriert")
    ax.plot(tabelle["vorhergesagt"], tabelle["beobachtet"], marker="o", label="Modell im Test")
    ax.set_xlabel("vorhergesagte Wahrscheinlichkeit (Mittel der Gruppe)")
    ax.set_ylabel("beobachtete Häufigkeit")
    ax.set_title("Kalibrierung im Testzeitraum")
    ax.legend(frameon=False)
    return tabelle

In [ ]:
SEMESTER = [("2024-10-14", "2025-02-14"), ("2025-04-22", "2025-07-25"), ("2025-10-20", "2026-02-13"), ("2026-04-20", "2026-07-24")]
MERKMALE_A = ["log_vorlauf", "pruefungsphase", "freitag", "spaet", "mitglied", "quote_bisher", "erstes_mal"]   # ▶ dazu der Typ als Anzeiger


def merkmale_A(daten):
    """Eine Zeile je Anmeldung mit Veranstaltung, Person und den Merkmalen des Startmodells."""
    a = (daten["anmeldungen"].merge(daten["veranstaltungen"], on="veranstaltung", how="left")
         .merge(daten["personen"], on="person", how="left"))
    a["vorlauf"] = (a["datum"] - a["angemeldet_am"]).dt.days
    a["log_vorlauf"] = np.log1p(a["vorlauf"])
    in_vorlesung = np.zeros(len(a), dtype=bool)
    for beginn, ende in SEMESTER:
        in_vorlesung |= a["datum"].between(pd.Timestamp(beginn), pd.Timestamp(ende)).to_numpy()
    a["pruefungsphase"] = (~in_vorlesung).astype(int)
    a["freitag"] = (a["datum"].dt.dayofweek == 4).astype(int)
    a["spaet"] = (a["beginn"].astype(str) >= "20:00").astype(int)    # ▶ später Beginn: ab 20 Uhr
    # bisherige Teilnahmequote nur aus früheren Anmeldungen derselben Person: (bisher erschienen + 1) / (bisher angemeldet + 2)
    a = a.sort_values(["person", "datum", "beginn", "anmeldung"])
    bisher_angemeldet = a.groupby("person").cumcount()
    bisher_erschienen = a.groupby("person")["erschienen"].cumsum() - a["erschienen"].fillna(0)
    a["quote_bisher"] = (bisher_erschienen + 1) / (bisher_angemeldet + 2)
    a["erstes_mal"] = (bisher_angemeldet == 0).astype(int)
    # 1, wenn in der Veranstaltung Anmeldungen mit und ohne Erinnerung vorkommen
    a["beide_gruppen"] = (a.groupby("veranstaltung")["erinnerung"].transform("nunique") == 2).astype(int)
    return a.sort_values("anmeldung").reset_index(drop=True)


def matrix_A(d):
    """Merkmale als Zahlenmatrix: Typ als Anzeiger (Vortrag als Bezug), dazu MERKMALE_A."""
    typ = pd.get_dummies(pd.Categorical(d["typ"], categories=["Vortrag", "Workshop", "Stammtisch", "Exkursion", "Hackathon"]), dtype=float).iloc[:, 1:]
    return np.column_stack([typ.to_numpy(), d[MERKMALE_A].to_numpy(dtype=float)])


MERKMALE_B = ["schwingung_rel", "temp_abw"]   # ▶ Merkmale des Startmodells


def merkmale_B(daten):
    """Eine Zeile je Maschinentag mit Schwingung relativ zum Median der 20 Vortage, Temperaturabweichung und Regel des Warnsystems."""
    d = daten["maschinentage"].merge(daten["maschinen"][["maschine", "typ", "pilot", "pilot_start"]], on="maschine", how="left")
    d = d.sort_values(["maschine", "datum"]).reset_index(drop=True)
    vortage = lambda s: s.shift(1).rolling(20, min_periods=10).median()
    d["schwingung_rel"] = d["vibration_mm_s"] / d.groupby("maschine")["vibration_mm_s"].transform(vortage)
    d["temp_abw"] = d["lagertemp_c"] - d.groupby("maschine")["lagertemp_c"].transform(vortage)
    d["regel_alarm"] = (d["schwingung_rel"] > 1.35).astype(int)       # Regel des Warnsystems, aus der Schwingung nachgerechnet
    d["pilotbetrieb"] = (d["pilot"] == 1) & (d["datum"] >= d["pilot_start"])
    return d


def regel_kennzahlen(daten, ab=None):
    """Recall, Precision und Alarmquote der Regel des Warnsystems für ausfall_morgen, ohne Pilotbetrieb, auf Wunsch ab einem Datum."""
    d = merkmale_B(daten)
    d = d[~d["pilotbetrieb"] & d["ausfall_morgen"].notna()]
    if ab is not None:
        d = d[d["datum"] >= pd.Timestamp(ab)]
    treffer = int(((d["regel_alarm"] == 1) & (d["ausfall_morgen"] == 1)).sum())
    return {"Recall": treffer / max(int(d["ausfall_morgen"].sum()), 1), "Precision": treffer / max(int(d["regel_alarm"].sum()), 1),
            "Alarmquote": float(d["regel_alarm"].mean()), "Tage": len(d)}

Das **Startmodell** ist eine logistische Regression. In Track A sagt sie voraus, wer zu einer Veranstaltung erscheint, mit Typ, Logarithmus des Vorlaufs, Prüfungsphase, Freitag, spätem Beginn, Mitgliedschaft, bisheriger Teilnahmequote und erstem Mal. Basis zum Vergleich ist die bisherige Teilnahmequote der Person allein. In Track B sagt sie voraus, ob eine Maschine am nächsten Produktionstag mindestens 60 Minuten ungeplant stillsteht, mit der Schwingung relativ zum Median der 20 Vortage und der Abweichung der Lagertemperatur. Basis ist die Regel des Warnsystems.

**Ausgeschlossen** sind Zeilen, in denen die Maßnahme selbst das Ergebnis verändern kann. In Track A sind das alle Anmeldungen mit Erinnerung: Das Modell soll beschreiben, wer ohne Erinnerung kommt, und die Wirkung der Erinnerung schätzt Analyse 2. In Track B ist das der Pilotbetrieb: Nach einem Alarm folgt eine Kurzinspektion, die einen Ausfall am nächsten Tag verhindern kann.

In [ ]:
# Startmodell und Basis im zeitlichen Test
if TRACK == "A":
    tabelle = merkmale_A(daten)
    tabelle = tabelle[(tabelle["erinnerung"] != 1) & tabelle["erschienen"].notna()].dropna(subset=MERKMALE_A)   # ohne Erinnerung
    training, test = zeitlich_teilen(tabelle, "datum", PLAN["split"])
    ziel, X_training, X_test = "erschienen", matrix_A(training), matrix_A(test)
    basis_name, basis_test = "bisherige Teilnahmequote allein", test["quote_bisher"]
else:
    tabelle = merkmale_B(daten)
    tabelle = tabelle[~tabelle["pilotbetrieb"]].dropna(subset=MERKMALE_B + ["ausfall_morgen"])                    # ohne Pilotbetrieb
    training, test = zeitlich_teilen(tabelle, "datum", PLAN["split"])
    ziel, X_training, X_test = "ausfall_morgen", training[MERKMALE_B].to_numpy(), test[MERKMALE_B].to_numpy()
    basis_name, basis_test = "Regel des Warnsystems", test["regel_alarm"]
modell = LogisticRegression(C=1e6, max_iter=10_000, random_state=SEED).fit(X_training, training[ziel].astype(int))
test["p"] = modell.predict_proba(X_test)[:, 1]
auc_modell = auc_mit_intervall(test[ziel], test["p"], SEED)
auc_basis = auc_mit_intervall(test[ziel], basis_test, SEED)
print(f"Training {len(training)} Zeilen bis {training['datum'].max():%d.%m.%Y}, Test {len(test)} Zeilen ab {test['datum'].min():%d.%m.%Y}")
print(f"Quote von {ziel} im Test: {test[ziel].mean():.3f}")
print(f"AUC Startmodell: {auc_modell[0]:.3f} [{auc_modell[1]:.3f}; {auc_modell[2]:.3f}]")
print(f"AUC Basis ({basis_name}): {auc_basis[0]:.3f} [{auc_basis[1]:.3f}; {auc_basis[2]:.3f}]")

In [ ]:
# Kalibrierung des Startmodells im Test
tabelle_kalibrierung = kalibrierung(test[ziel], test["p"])
print(tabelle_kalibrierung.round(3).to_string())
plt.show()

In [ ]:
# Eine Größe für die Entscheidung
if TRACK == "A":
    # erwartete gegen tatsächliche Teilnahme je Veranstaltung im Testzeitraum (Anmeldungen ohne Erinnerung)
    je_v = test.groupby("veranstaltung").agg(erwartet=("p", "sum"), tatsaechlich=("erschienen", "sum"), typ=("typ", "first"))
    fig, ax = plt.subplots(figsize=(5, 4.5), layout="constrained")
    for typ, teil in je_v.groupby("typ"):
        ax.scatter(teil["erwartet"], teil["tatsaechlich"], label=typ, s=25)
    grenze = je_v[["erwartet", "tatsaechlich"]].max().max() * 1.05
    ax.plot([0, grenze], [0, grenze], color="grey", linestyle="--")
    ax.set_xlabel("erwartete Teilnahme (Summe der Wahrscheinlichkeiten)")
    ax.set_ylabel("tatsächlich erschienen")
    ax.set_title("Je Veranstaltung im Testzeitraum")
    ax.legend(frameon=False, fontsize=8)
    print(f"{len(je_v)} Veranstaltungen; mittlere Abweichung tatsächlich minus erwartet: {(je_v['tatsaechlich'] - je_v['erwartet']).mean():.2f} Personen,"
          f" mittlerer Betrag: {(je_v['tatsaechlich'] - je_v['erwartet']).abs().mean():.2f}")
else:
    kennzahlen = regel_kennzahlen(daten, ab=PLAN["split"])
    print("Regel des Warnsystems im Test:", {k: round(v, 3) for k, v in kennzahlen.items()})
    im_pilot = merkmale_B(daten).query("pilotbetrieb")
    print(f"Pilotbetrieb: {int(im_pilot['alarm'].sum())} Alarme laut Spalte alarm, {int(im_pilot['regel_alarm'].sum())} nachgerechnet,"
          f" gleich an {int((im_pilot['alarm'] == im_pilot['regel_alarm']).sum())} von {len(im_pilot)} Tagen")
    je_typ = []
    for typ, teil in test.groupby("typ"):
        if teil[ziel].nunique() == 2:
            je_typ.append({"Typ": typ, "Tage": len(teil), "Ausfälle": int(teil[ziel].sum()), "AUC": round(roc_auc_score(teil[ziel], teil["p"]), 3)})
    je_typ = pd.DataFrame(je_typ).sort_values("AUC")
    print(je_typ.to_string(index=False))
    fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
    ax.barh(je_typ["Typ"], je_typ["AUC"])
    ax.axvline(0.5, color="grey", linestyle="--")
    ax.axvline(auc_modell[0], color="black", linewidth=0.8)
    ax.set_xlim(0.4, 1)
    ax.set_xlabel("AUC im Test (schwarz: alle Maschinen)")
    ax.set_title("Vorhersagbarkeit je Maschinentyp")
plt.show()

## Schritt 4 · Analyse 2: Wirkung

*Ergebnis:* Schätzung mit 95-%-Intervall aus einem passenden Design und mindestens eine geprüfte Annahme.

**Track A** wertet den Versuch mit der Erinnerung aus. `versuch_A(daten, zeilen, bloecke, kovariate)` schätzt den Unterschied der Teilnahme mit und ohne Erinnerung in Prozentpunkten mit einer linearen Regression (statsmodels). `zeilen` wählt die Anmeldungen als Bedingung für `DataFrame.query` auf der Tabelle aus `merkmale_A`; mit `bloecke=True` kommt die Veranstaltung als fester Effekt dazu, und die Standardfehler sind nach Veranstaltung geklumpt; `kovariate` nimmt eine Spalte als Kovariate auf. Die Funktion prüft auch die Zuteilung: Anteil mit Erinnerung und Binomialtest gegen 50 %.

**Track B** wertet den Pilot aus. `monatsverlauf_B(daten)` zeichnet den ungeplanten Stillstand je Maschinentag im Monatsmittel für die Pilotmaschinen und die übrigen. `did(daten, vor, nach_ab, seed)` rechnet die Differenz-in-Differenzen: je Maschine das Mittel je Maschinentag in der Vorperiode und ab `nach_ab`, Differenz nachher minus vorher, Pilotmaschinen minus übrige, mit einem Intervall aus 2.000 Bootstrap-Ziehungen über Maschinen (geschichtet nach Pilot und Vergleich). Ein **Placebo** mit einem erfundenen Start vor dem Pilot prüft, ob die Methode auch dort eine Wirkung findet, wo es noch keine geben dürfte.

In [ ]:
def versuch_A(daten, zeilen, bloecke, kovariate):
    """Unterschied der Teilnahme mit und ohne Erinnerung in Prozentpunkten, mit 95-%-Intervall, und Prüfung der Zuteilung."""
    d = merkmale_A(daten)
    d = d[d["erinnerung"].isin([0, 1]) & d["erschienen"].notna()].query(zeilen)
    if kovariate:
        d = d.dropna(subset=[kovariate])
    formel = "erschienen ~ erinnerung" + (" + C(veranstaltung)" if bloecke else "") + (f" + {kovariate}" if kovariate else "")
    if bloecke:
        fit = smf.ols(formel, data=d).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d["veranstaltung"])[0]})
    else:
        fit = smf.ols(formel, data=d).fit(cov_type="HC1")
    b, se = fit.params["erinnerung"], fit.bse["erinnerung"]
    k = int(d["erinnerung"].sum())
    return {"Anmeldungen": len(d), "Veranstaltungen": d["veranstaltung"].nunique(), "Anteil mit Erinnerung": k / len(d),
            "Quote ohne": float(d.loc[d["erinnerung"] == 0, "erschienen"].mean()), "Quote mit": float(d.loc[d["erinnerung"] == 1, "erschienen"].mean()),
            "p Binomialtest 50 %": float(stats.binomtest(k, len(d), 0.5).pvalue),
            "Schätzung": 100 * b, "unten": 100 * (b - 1.96 * se), "oben": 100 * (b + 1.96 * se), "Formel": formel}


def monatsverlauf_B(daten):
    """Monatsmittel des ungeplanten Stillstands je Maschinentag, Pilotmaschinen gegen übrige; zeichnet die Abbildung."""
    d = daten["maschinentage"].merge(daten["maschinen"][["maschine", "pilot"]], on="maschine", how="left")
    d["monat"] = d["datum"].dt.to_period("M").dt.to_timestamp()
    tafel = d.pivot_table(index="monat", columns="pilot", values="stillstand_min", aggfunc="mean").rename(columns={0: "Vergleich", 1: "Pilot"})
    start = daten["maschinen"]["pilot_start"].min()
    fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
    ax.axvspan(pd.Timestamp("2025-10-01"), pd.Timestamp("2025-12-31"), color="grey", alpha=0.15, label="Auswahlquartal (4. Quartal 2025)")
    ax.plot(tafel.index, tafel["Pilot"], marker="o", label="Pilotmaschinen")
    ax.plot(tafel.index, tafel["Vergleich"], marker="o", label="übrige Maschinen")
    ax.axvline(start, color="black", linestyle="--", label=f"Pilotstart {start:%d.%m.%Y}")
    ax.set_ylabel("ungeplanter Stillstand\nMinuten je Maschinentag")
    ax.set_title("Monatsmittel je Maschinentag")
    ax.legend(frameon=False, fontsize=8)
    return tafel


def did(daten, vor, nach_ab, seed, bis=None, ziehungen=2000):
    """Differenz-in-Differenzen in Minuten je Maschinentag mit geschichtetem Bootstrap über Maschinen."""
    d = daten["maschinentage"]
    if bis is not None:
        d = d[d["datum"] <= pd.Timestamp(bis)]
    vorher = d[d["datum"].between(pd.Timestamp(vor[0]), pd.Timestamp(vor[1]))].groupby("maschine")["stillstand_min"].mean()
    nachher = d[d["datum"] >= pd.Timestamp(nach_ab)].groupby("maschine")["stillstand_min"].mean()
    tafel = pd.DataFrame({"vorher": vorher, "nachher": nachher}).reindex(daten["maschinen"]["maschine"])
    tafel["differenz"] = tafel["nachher"] - tafel["vorher"]
    ohne = tafel.index[tafel["differenz"].isna()]
    if len(ohne):
        print(f"Hinweis: {', '.join(ohne)} ohne Tage in einem der Zeiträume, nicht berücksichtigt")
    pilot_kennung = set(daten["maschinen"].loc[daten["maschinen"]["pilot"] == 1, "maschine"])
    ist_pilot = tafel.index.isin(pilot_kennung)
    pilot = tafel.loc[ist_pilot, "differenz"].dropna().to_numpy()
    rest = tafel.loc[~ist_pilot, "differenz"].dropna().to_numpy()
    schaetzung = pilot.mean() - rest.mean()
    rng = np.random.default_rng(seed)
    werte = np.empty(ziehungen)
    for i in range(ziehungen):
        werte[i] = rng.choice(pilot, len(pilot)).mean() - rng.choice(rest, len(rest)).mean()
    basis = tafel.loc[ist_pilot, "nachher"].mean() - schaetzung
    unten, oben = np.quantile(werte, [0.025, 0.975])
    return {"Schätzung": float(schaetzung), "unten": float(unten), "oben": float(oben), "Basis": float(basis),
            "relativ": float(schaetzung / basis), "Ziehungen": werte, "Tafel": tafel}

In [ ]:
# Analyse 2 nach PLAN
if TRACK == "A":
    wirkung = versuch_A(daten, PLAN["zeilen"], PLAN["bloecke"], PLAN["kovariate"])
    print(f"Formel: {wirkung['Formel']}")
    print(f"{wirkung['Anmeldungen']} Anmeldungen in {wirkung['Veranstaltungen']} Veranstaltungen, Anteil mit Erinnerung {wirkung['Anteil mit Erinnerung']:.3f},"
          f" Binomialtest gegen 50 %: p = {wirkung['p Binomialtest 50 %']:.3f}")
    print(f"Erinnerung: {wirkung['Schätzung']:+.1f} Prozentpunkte [{wirkung['unten']:.1f}; {wirkung['oben']:.1f}]")
    fig, ax = plt.subplots(figsize=(5, 4), layout="constrained")
    ax.bar(["ohne Erinnerung", "mit Erinnerung"], [wirkung["Quote ohne"], wirkung["Quote mit"]], color=["C0", "C1"])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Anteil erschienen")
    ax.set_title(f"Erinnerung: {wirkung['Schätzung']:+.1f} Prozentpunkte [{wirkung['unten']:.1f}; {wirkung['oben']:.1f}]")
else:
    wirkung = did(daten, PLAN["vorperiode"], PLAN["nach_ab"], SEED)
    print(f"Vorperiode {PLAN['vorperiode'][0]} bis {PLAN['vorperiode'][1]}, nachher ab {PLAN['nach_ab']}")
    print(f"Differenz-in-Differenzen: {wirkung['Schätzung']:+.1f} Minuten je Maschinentag [{wirkung['unten']:.1f}; {wirkung['oben']:.1f}],"
          f" relativ {100 * wirkung['relativ']:+.0f} % gegenüber der Basis {wirkung['Basis']:.1f} Minuten")
    tafel_monate = monatsverlauf_B(daten)
plt.show()

In [ ]:
# Eine Annahme prüfen
if TRACK == "A":
    # Gleichgewicht der Gruppen: Merkmale, die vor der Erinnerung feststanden, im Mittel je Gruppe
    d = merkmale_A(daten)
    d = d[d["erinnerung"].isin([0, 1]) & d["erschienen"].notna()].query(PLAN["zeilen"])
    print(d.groupby("erinnerung")[["quote_bisher", "erstes_mal", "mitglied", "vorlauf"]].mean().round(3).to_string())
    print(d.groupby(d["datum"].dt.to_period("M"))["erinnerung"].agg(["size", "mean"]).round(3).rename(columns={"size": "Anmeldungen", "mean": "Anteil mit Erinnerung"}).to_string())
else:
    # Placebo: erfundener Start am 1. April 2025, nur Daten bis 30. September 2025
    placebo = did(daten, ("2024-10-01", "2025-03-31"), "2025-04-01", SEED, bis="2025-09-30")
    print(f"Placebo (Start 01.04.2025): {placebo['Schätzung']:+.1f} Minuten je Maschinentag [{placebo['unten']:.1f}; {placebo['oben']:.1f}]")

## Schritt 5 · Entscheidung und Vorlage

*Ergebnis:* Entscheidungsrechnung mit Unsicherheit, `entscheidung.png` und eine Entscheidungsvorlage von höchstens zwei Seiten, Empfehlung zuerst.

Die Entscheidungsrechnung verbindet beide Analysen mit den Annahmen der Auftraggeber aus der `LIESMICH.md`.

**Track A:** `anmeldegrenze(p, plaetze, risiko, seed)` zieht in 4.000 Simulationen je Zahl der Anmeldungen N die Anmeldungen aus einem Pool vorhergesagter Erscheinwahrscheinlichkeiten (Startmodell, Anmeldungen desselben Typs im Testzeitraum) und lässt jede Person mit ihrer Wahrscheinlichkeit erscheinen. Das Risiko für N ist der Anteil der Simulationen mit mehr Erschienenen als Plätzen. Ergebnis ist das größte N mit einem Risiko von höchstens `risiko`. Mit Erinnerung steigen die Wahrscheinlichkeiten um `plus`, die Wirkung aus Analyse 2 in Prozentpunkten: die Schätzung oder, vorsichtiger, die obere Intervallgrenze (`PLAN["aufschlag"]`), denn eine größere Wirkung füllt den Raum stärker. Mit `sd_veranstaltung` größer 0 bekommt jede Veranstaltung der Simulation eine gemeinsame Verschiebung auf der Logit-Skala, etwa für schlechtes Wetter oder ein beliebtes Thema. So rechnet auch das Projekthandbuch in Abschnitt 5.3.

**Track B:** `rollout_B(daten, ziehungen, seed)` rechnet je Maschine das Netto in Euro je Jahr: eingesparter Stillstand (relative Wirkung aus `did` mal Grundstillstand je Jahr mal 20 Euro) minus Kurzinspektionen (Alarme je Jahr mal 400 Euro) minus 5.000 Euro für das Warnsystem. Jede Bootstrap-Ziehung aus `did` gibt eine relative Wirkung und damit ein Netto, daraus folgen Mittel, 95-%-Intervall und die Wahrscheinlichkeit für ein Netto über 0. Die Vorgabe nimmt für alle Maschinen dieselbe relative Wirkung an. Prüft diese Annahme mit Analyse 1, etwa über die Vorhersagbarkeit je Maschinentyp, und mit `sd_maschine` größer 0.

In [ ]:
def anmeldegrenze(p, plaetze, risiko, seed, plus=0.0, sd_veranstaltung=0.0, simulationen=4000):
    """Größtes N mit P(mehr Erschienene als Plätze) <= risiko und das Risiko für jedes N bis 3 mal Plätze.
    p: Pool vorhergesagter Erscheinwahrscheinlichkeiten; plus: Aufschlag in Prozentpunkten als Anteil (0,054 für 5,4 Punkte);
    sd_veranstaltung: gemeinsame Verschiebung je Veranstaltung auf der Logit-Skala. Kumulierte Summen rechnen alle N in einem Lauf."""
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    rng = np.random.default_rng(seed)
    logit = np.log(p / (1 - p))[rng.integers(0, len(p), size=(simulationen, 3 * plaetze))]
    q = np.clip(1 / (1 + np.exp(-(logit + rng.normal(0, sd_veranstaltung, size=(simulationen, 1))))) + plus, 0, 1)
    erschienen = np.cumsum(rng.random(q.shape) < q, axis=1)
    kurve = pd.Series((erschienen > plaetze).mean(axis=0), index=np.arange(1, 3 * plaetze + 1), name="Risiko")
    erlaubt = kurve.index[kurve <= risiko]
    return (int(erlaubt.max()) if len(erlaubt) else 0), kurve


def rollout_B(daten, ziehungen, seed, sd_maschine=0.0):
    """Netto in Euro je Maschine und Jahr; ziehungen ist das Ergebnis von did (Ziehungen und Basis)."""
    d = merkmale_B(daten)
    tage_jahr = d["datum"].nunique() / 2                          # Produktionstage je Jahr: zwei Jahre im Paket
    start = daten["maschinen"]["pilot_start"].min()
    grund = d[d["datum"] < start].groupby("maschine")["stillstand_min"].mean() * tage_jahr
    alarme = d.groupby("maschine")["regel_alarm"].sum() / 2
    m = daten["maschinen"].set_index("maschine")[["typ", "pilot"]].join(grund.rename("grund_min_jahr")).join(alarme.rename("alarme_jahr"))
    relativ = np.asarray(ziehungen["Ziehungen"]) / ziehungen["Basis"]
    rng = np.random.default_rng(seed)
    relativ = relativ[:, None] + (rng.normal(0, sd_maschine, size=(len(relativ), len(m))) if sd_maschine > 0 else 0)
    netto = (-relativ * m["grund_min_jahr"].to_numpy() * KOSTEN["minute"]
             - m["alarme_jahr"].to_numpy() * KOSTEN["inspektion"] - KOSTEN["system"])
    m["netto_mittel"] = netto.mean(axis=0)
    m["netto_unten"], m["netto_oben"] = np.quantile(netto, [0.025, 0.975], axis=0)
    m["p_gewinn"] = (netto > 0).mean(axis=0)
    return m.sort_values("netto_mittel", ascending=False)

In [ ]:
# Annahmen der Auftraggeber laut LIESMICH.md
KOSTEN = {"minute": 20, "inspektion": 400, "system": 5000}             # Track B: Euro je Minute Stillstand, je Kurzinspektion, je Maschine und Jahr
RAEUME = {25: ("Exkursion", 0.01), 30: ("Workshop", 0.05), 40: ("Stammtisch", 0.05), 60: ("Vortrag", 0.05), 120: ("Vortrag", 0.05)}
# ▶ RAEUME (Track A): Plätze -> (Typ für den Pool der Wahrscheinlichkeiten, erlaubtes Risiko). Passt der Pool zur Raumgröße?

In [ ]:
# Entscheidungsrechnung, Ergebnistabelle und entscheidung.png
if TRACK == "A":
    pool = test[["typ", "p"]]
    plus = (wirkung["oben"] if PLAN["aufschlag"] == "obere Grenze" else wirkung["Schätzung"]) / 100   # Prozentpunkte als Anteil
    print(f"Aufschlag durch die Erinnerung: {100 * plus:.1f} Prozentpunkte ({PLAN['aufschlag']})")
    zeilen, kurven = [], {}
    for plaetze, (typ, risiko) in RAEUME.items():
        p_typ = pool.loc[pool["typ"] == typ, "p"].to_numpy()
        n_ohne, kurve_ohne = anmeldegrenze(p_typ, plaetze, risiko, SEED)
        n_mit, kurve_mit = anmeldegrenze(p_typ, plaetze, risiko, SEED, plus=plus)
        kurven[plaetze] = (kurve_ohne, kurve_mit)
        zeilen.append({"Plätze": plaetze, "Typ im Pool": typ, "Pool": len(p_typ), "Risiko höchstens": risiko,
                       "Anmeldungen ohne Erinnerung": n_ohne, "Anmeldungen mit Erinnerung": n_mit})
    ergebnis = pd.DataFrame(zeilen)
    print(ergebnis.to_string(index=False))
    regel_3 = pd.DataFrame({"Wert": ["Schätzung", "untere Intervallgrenze"], "Prozentpunkte": [round(wirkung["Schätzung"], 1), round(wirkung["unten"], 1)]})
    regel_3["mindestens 3 Prozentpunkte"] = np.where(regel_3["Prozentpunkte"] >= 3, "ja", "nein")
    print()
    print(regel_3.to_string(index=False))
    fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
    for i, (plaetze, (kurve_ohne, kurve_mit)) in enumerate(kurven.items()):
        farbe = f"C{i}"
        ax.plot(kurve_ohne.index, kurve_ohne, color=farbe, label=f"{plaetze} Plätze")
        ax.plot(kurve_mit.index, kurve_mit, color=farbe, linestyle="--")
    ax.axhline(0.05, color="grey", linewidth=0.8)
    ax.axhline(0.01, color="grey", linewidth=0.8, linestyle=":")
    ax.set_xlabel("Zahl der Anmeldungen")
    ax.set_ylabel("Risiko: mehr Erschienene als Plätze")
    ax.set_title("Anmeldegrenze je Raumgröße (durchgezogen ohne, gestrichelt mit Erinnerung)")
    ax.legend(frameon=False, fontsize=8)
else:
    ergebnis = rollout_B(daten, wirkung, SEED)
    print(ergebnis.round({"grund_min_jahr": 0, "alarme_jahr": 1, "netto_mittel": 0, "netto_unten": 0, "netto_oben": 0, "p_gewinn": 3}).to_string())
    reihe = ergebnis.sort_values("netto_mittel")
    fig, ax = plt.subplots(figsize=(7, 6), layout="constrained")
    farben = np.where(reihe["pilot"] == 1, "C1", "C0")
    ax.errorbar(reihe["netto_mittel"], np.arange(len(reihe)), xerr=[reihe["netto_mittel"] - reihe["netto_unten"], reihe["netto_oben"] - reihe["netto_mittel"]],
                fmt="none", ecolor="grey", capsize=3)
    ax.scatter(reihe["netto_mittel"], np.arange(len(reihe)), c=farben, zorder=3)
    ax.set_yticks(np.arange(len(reihe)), [f"{k} {t}" for k, t in zip(reihe.index, reihe["typ"])])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Netto in Euro je Jahr (Mittel und 95-%-Intervall)")
    ax.set_title("Rollout des Warnsystems je Maschine (orange: Pilot)")
plt.savefig("entscheidung.png", dpi=150, bbox_inches="tight")
plt.show()

### Gerüst der Entscheidungsvorlage

Kopiert das Gerüst nach `ENTSCHEIDUNGSVORLAGE.md` und füllt es aus. Höchstens zwei Seiten, **Empfehlung zuerst**, jede Zahl mit Intervall, jede Annahme benannt. Schreibt für die Werkleitung oder den Vorstand, nicht für Data Scientists.

```markdown
# Entscheidungsvorlage: <Titel>
**Für:** <Werkleitung / Vorstand> · **Von:** <Namen> · **Paket:** <Track und Nummer> · **Stand:** <Datum>

## Empfehlung
<Ein Satz: was, für wen, unter welcher Bedingung.>

## Begründung in Zahlen
- <Wirkung mit 95-%-Intervall und Bezugsgröße>
- <Entscheidungsgröße mit Intervall, etwa Netto je Jahr oder Anmeldegrenze>

## Die zwei Analysen
**Analyse 1 · Vorhersage:** <drei Sätze>
**Analyse 2 · Wirkung:** <drei Sätze>

## Annahmen und Risiken
- <Annahme, wie geprüft, was passiert, wenn sie nicht gilt>

## Nächste Schritte
- <Schritt mit Zuständigkeit und Zeitpunkt>

## Anhang: Datenqualität
| Problem | Umfang | Umgang | Folge |
```

### Checkliste vor dem Einreichen
- [ ] README und Notebook, das von oben nach unten läuft
- [ ] Datenqualitätsbericht
- [ ] Analyseplan mit Datum vor den Ergebnissen
- [ ] Analyse 1 mit zeitlichem Test und Intervall
- [ ] Analyse 2 mit Intervall und geprüften Annahmen
- [ ] Entscheidungsvorlage von höchstens zwei Seiten
- [ ] Präsentation für zehn Minuten
- [ ] drei Terminvorschläge